In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import difflib

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

# ============================================================
# Configuration
# ============================================================

NAMESPACE = "web01"

PRODUCTION_ENTRYPOINT = Path("/etc/nftables.conf")
TEST_ENTRYPOINT = Path("/tmp/nftables-zones-test.conf")

VARIABLES_INCLUDE = "/etc/nft/web01/variables/*.nft"
CONSTANTS_INCLUDE = "/etc/nft/web01/constants/*.nft"
ZONES_INCLUDE = "/etc/nft/web01/zones/*.nft"

PROTOCOLS = {
    "TCP": {
        "source": Path("/etc/nft/web01/rules/10-tcp-networks.nft"),
        "staging": Path("/tmp/10-tcp-network.nft"),
    },
    "UDP": {
        "source": Path("/etc/nft/web01/rules/08-udp-networks.nft"),
        "staging": Path("/tmp/08-udp-networks.nft"),
    },
    "ICMP": {
        "source": Path("/etc/nft/web01/rules/06-icmp-networks.nft"),
        "staging": Path("/tmp/06-icmp-network.nft"),
    },
}

def rule_group(label, *definitions):
    return {"label": label, "definitions": list(definitions)}

def definition(name, *fields):
    return {"name": name, "fields": list(fields)}

def paired_group(zone, suffix, fields01, fields02, label="R01/R02"):
    return rule_group(
        label,
        definition(f"Z{zone}_{suffix}_R01", *fields01),
        definition(f"Z{zone}_{suffix}_R02", *fields02),
    )

COMMON_FIELDS = ("interface", "source", "destination")
STANDARD_R01 = ("$zone", *COMMON_FIELDS, "port")
STANDARD_R02 = (*COMMON_FIELDS, "port")
TRANSPORT_R01 = (
    "$zone", "interface", "source", "source_port", "destination_port"
)
TRANSPORT_R02 = ("interface", "source", "source_port", "destination_port")
ICMP_R01 = ("$zone", *COMMON_FIELDS, "icmp_type")
ICMP_R02 = (*COMMON_FIELDS, "icmp_type")

# Each zone lists only its supported protocols. A protocol contains one or
# more rule groups; each definition declares its tuple fields in source order.
# Add new zones/protocols/groups here rather than relying on naming conventions.
ZONE_CONFIG = {
    "01": {
        "TCP": {"label": "TCPv4", "groups": [
            paired_group("01", "TCP4", STANDARD_R01, STANDARD_R02),
        ]},
        "UDP": {"label": "UDPv4", "groups": [
            paired_group("01", "UDP4", STANDARD_R01, STANDARD_R02),
        ]},
        "ICMP": {"label": "ICMPv4", "groups": [
            paired_group("01", "ICMP4", ICMP_R01, ICMP_R02),
        ]},
    },
    "02": {
        "TCP": {"label": "TCPv4", "groups": [
            paired_group("02", "TCP4", STANDARD_R01, STANDARD_R02),
        ]},
        "UDP": {"label": "UDPv4", "groups": [
            paired_group("02", "UDP4", STANDARD_R01, STANDARD_R02),
        ]},
        "ICMP": {"label": "ICMPv4", "groups": [
            paired_group("02", "ICMP4", ICMP_R01, ICMP_R02),
        ]},
    },
    "03": {
        "TCP": {"label": "TCPv4", "groups": [
            paired_group("03", "TCP4", TRANSPORT_R01, TRANSPORT_R02),
        ]},
        "UDP": {"label": "UDPv4", "groups": [
            paired_group("03", "UDP4", TRANSPORT_R01, TRANSPORT_R02),
        ]},
        "ICMP": {"label": "ICMPv4", "groups": [
            paired_group(
                "03", "ICMP4",
                ("$zone", "interface", "destination", "icmp_type"),
                ("interface", "destination", "icmp_type"),
            ),
        ]},
    },
    "11": {
        "TCP": {"label": "TCPv6", "groups": [
            paired_group("11", "TCP6", STANDARD_R01, STANDARD_R02),
        ]},
        "UDP": {"label": "UDPv6", "groups": [
            paired_group("11", "UDP6", STANDARD_R01, STANDARD_R02),
        ]},
        "ICMP": {"label": "ICMPv6", "groups": [
            rule_group(
                "request/reply",
                definition(
                    "Z11_icmp6req", "interface", "source", "destination"
                ),
                definition(
                    "Z11_icmp6reply", "interface", "source", "destination"
                ),
            ),
        ]},
    },
    "12": {
        "TCP": {"label": "TCPv6", "groups": [
            paired_group("12", "TCP6", STANDARD_R01, STANDARD_R02),
        ]},
        "UDP": {"label": "UDPv6", "groups": [
            paired_group("12", "UDP6", STANDARD_R01, STANDARD_R02),
        ]},
        "ICMP": {"label": "ICMPv6", "groups": [
            rule_group(
                "request/reply",
                definition(
                    "Z12_icmp6req", "interface", "source", "destination"
                ),
                definition(
                    "Z12_icmp6reply", "interface", "source", "destination"
                ),
            ),
        ]},
    },
    "13": {
        "TCP": {"label": "TCPv6", "groups": [
            paired_group("13", "TCP6", TRANSPORT_R01, TRANSPORT_R02),
        ]},
        "UDP": {"label": "UDPv6", "groups": [
            paired_group("13", "UDP6", TRANSPORT_R01, TRANSPORT_R02),
        ]},
    },
}

ZONE_IDS = tuple(ZONE_CONFIG)
ALLOWED_VERDICTS = {"accept", "drop", "reject"}
original_sources = {}
staged_sources = {}
rules_by_zone_protocol = {}


In [ ]:
# ============================================================
# Read source files
# ============================================================

def read_source(protocol):
    path = PROTOCOLS[protocol]["source"]

    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot read {protocol} source file: {path}"
        )

    return path.read_text()


In [ ]:
# ============================================================
# Parse definitions, including empty definitions
# ============================================================

def parse_definition(source, name):
    pattern = (
        r"(?m)^[ \t]*define[ \t]+"
        + re.escape(name)
        + r"[ \t]*=[ \t]*\{([^}]*)\}"
    )

    matches = list(re.finditer(pattern, source, flags=re.DOTALL))

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one definition of {name}; "
            f"found {len(matches)}."
        )

    entries = []

    for line_number, raw_line in enumerate(
        matches[0].group(1).splitlines(), start=1
    ):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        entry_match = re.fullmatch(
            r"(.+?)\s*:\s*([A-Za-z_][A-Za-z0-9_]*)\s*,?",
            line,
        )

        if not entry_match:
            raise ValueError(
                f"Cannot parse entry in {name}, body line "
                f"{line_number}: {raw_line!r}"
            )

        entries.append({
            "fields": [
                field.strip()
                for field in entry_match.group(1).strip().split(".")
            ],
            "verdict": entry_match.group(2).strip().lower(),
        })

    return entries

In [ ]:
# ============================================================
# Manifest helpers
# ============================================================

def zone_protocol_config(zone, protocol):
    try:
        return ZONE_CONFIG[zone][protocol]
    except KeyError as exc:
        raise ValueError(
            f"Zone {zone} does not define protocol {protocol}."
        ) from exc


def group_fields(group):
    definitions = group["definitions"]
    if not definitions:
        raise ValueError(f"Rule group {group['label']} has no definitions.")

    layouts = [
        tuple(field for field in item["fields"] if field != "$zone")
        for item in definitions
    ]
    if any(layout != layouts[0] for layout in layouts[1:]):
        raise ValueError(
            f"Rule group {group['label']} has incompatible tuple layouts."
        )
    return layouts[0]


In [ ]:
# ============================================================
# Load definitions described by the zone manifest
# ============================================================

def load_zone_protocol(source, protocol, zone):
    config = zone_protocol_config(zone, protocol)
    loaded_groups = []

    for group in config["groups"]:
        fields = group_fields(group)
        definitions = []

        for item in group["definitions"]:
            definitions.append(
                (item, parse_definition(source, item["name"]))
            )

        counts = {len(entries) for _, entries in definitions}
        if len(counts) != 1:
            raise ValueError(
                f"Zone {zone} {config['label']} group {group['label']}: "
                "definition entry counts differ."
            )

        rules = []
        for number, rows in enumerate(zip(
            *(entries for _, entries in definitions)
        ), start=1):
            rule = {}
            verdicts = []

            for (item, _), row in zip(definitions, rows):
                if len(row["fields"]) != len(item["fields"]):
                    raise ValueError(
                        f"Zone {zone} {config['label']} "
                        f"{item['name']} entry {number}: expected "
                        f"{len(item['fields'])} fields, found "
                        f"{row['fields']!r}."
                    )

                for field, value in zip(item["fields"], row["fields"]):
                    if field == "$zone":
                        if value != f"$Z{zone}":
                            raise ValueError(
                                f"{item['name']} entry {number}: expected "
                                f"$Z{zone}, found {value!r}."
                            )
                    else:
                        rule_value = rule.setdefault(field, value)
                        if rule_value != value:
                            raise ValueError(
                                f"Zone {zone} {config['label']} group "
                                f"{group['label']} entry {number}: "
                                "paired tuple fields differ."
                            )
                verdicts.append(row["verdict"])

            if len(set(verdicts)) != 1:
                raise ValueError(
                    f"Zone {zone} {config['label']} group {group['label']} "
                    f"entry {number}: paired verdicts differ."
                )
            rule["verdict"] = verdicts[0]
            rules.append(rule)

        loaded_groups.append(rules)

    return loaded_groups


In [ ]:
# ============================================================
# Initialize configured zones and protocols
# ============================================================

original_sources.clear()
staged_sources.clear()
rules_by_zone_protocol = {
    zone: {} for zone in ZONE_IDS
}

for protocol in PROTOCOLS:
    source = read_source(protocol)
    original_sources[protocol] = source

    for zone, config in ZONE_CONFIG.items():
        if protocol in config:
            rules_by_zone_protocol[zone][protocol] = (
                load_zone_protocol(source, protocol, zone)
            )

print("Zone editor configuration loaded successfully.")

for zone, config in ZONE_CONFIG.items():
    print(f"\nZone {zone}:")
    for protocol, protocol_config in config.items():
        count = sum(
            len(group_rules)
            for group_rules in rules_by_zone_protocol[zone][protocol]
        )
        print(f"  {protocol_config['label']}: {count} rules")

print(f"\nNamespace: {NAMESPACE}")
print(f"Test entry point: {TEST_ENTRYPOINT}")
print(f"Production entry point: {PRODUCTION_ENTRYPOINT}")


In [ ]:
# ============================================================
# Render definitions from manifest tuple layouts
# ============================================================

def render_definition(item, rules, zone):
    name = item["name"]
    fields = item["fields"]
    if not rules:
        return f"define {name} = {{}}"

    lines = [f"define {name} = {{"]
    for rule in rules:
        values = [
            f"$Z{zone}" if field == "$zone" else rule[field]
            for field in fields
        ]
        lines.append(
            f"    {' . '.join(values)} : {rule['verdict']},"
        )
    lines.append("}")
    return "\n".join(lines)


In [ ]:
# ============================================================
# Manifest-driven validation
# ============================================================

def validate_protocol(zone, protocol):
    errors = []
    config = zone_protocol_config(zone, protocol)
    groups = rules_by_zone_protocol[zone][protocol]

    for group, rules in zip(config["groups"], groups):
        fields = group_fields(group)
        seen = set()
        for number, rule in enumerate(rules, start=1):
            label = (
                f"Zone {zone} {config['label']} "
                f"{group['label']} rule {number}"
            )
            if set(rule) != set(fields) | {"verdict"}:
                errors.append(f"{label}: fields do not match the manifest.")
                continue

            for field in fields:
                value = rule[field]
                if not isinstance(value, str) or not value.strip():
                    errors.append(f"{label}: {field} is empty.")
                    continue
                if any(c in value for c in ("\n", "\r", ",", ";", "{", "}")):
                    errors.append(f"{label}: invalid delimiter in {field}.")
                if ":" in value:
                    errors.append(f"{label}: unexpected colon in {field}.")

                if field in ("port", "source_port", "destination_port"):
                    valid_port = (
                        (value.isdigit() and 1 <= int(value) <= 65535)
                        or bool(re.fullmatch(
                            r"\$[A-Za-z_][A-Za-z0-9_]*", value
                        ))
                        or bool(re.fullmatch(
                            r"[A-Za-z_][A-Za-z0-9_-]*", value
                        ))
                    )
                    if not valid_port:
                        errors.append(
                            f"{label}: invalid {field.replace('_', ' ')} "
                            f"{value!r}."
                        )
                elif field == "icmp_type" and not re.fullmatch(
                    r"[A-Za-z_][A-Za-z0-9_-]*", value
                ):
                    errors.append(f"{label}: invalid ICMP type {value!r}.")

            if rule["verdict"] not in ALLOWED_VERDICTS:
                errors.append(f"{label}: invalid verdict.")
            signature = tuple(rule.get(field, "") for field in fields)
            if signature in seen:
                errors.append(f"{label}: duplicate rule.")
            seen.add(signature)

    if len(groups) != len(config["groups"]):
        errors.append(
            f"Zone {zone} {config['label']}: loaded group count "
            "does not match the manifest."
        )
    return errors


In [ ]:
# ============================================================
# Editor widgets
# ============================================================

zone_selector = widgets.ToggleButtons(
    options=[(f"Zone {zone}", zone) for zone in ZONE_IDS],
    value=ZONE_IDS[0],
    description="Zone:",
)

protocol_selector = widgets.ToggleButtons(description="Protocol:")
group_selector = widgets.Dropdown(description="Rule group:")
rule_selector = widgets.Dropdown(description="Rule:")
field_inputs = {}
field_editor = widgets.HBox()

verdict_input = widgets.Dropdown(
    options=["accept", "drop", "reject"],
    value="accept",
    description="Verdict:",
    layout=widgets.Layout(width="170px"),
)

add_button = widgets.Button(description="Add", button_style="success")
update_button = widgets.Button(description="Update", button_style="primary")
delete_button = widgets.Button(description="Delete", button_style="danger")

table_output = widgets.Output()
status_output = widgets.Output()
workflow_output = widgets.Output()
diff_output = widgets.Output()

stage_button = widgets.Button(
    description="Check All Configured Zones",
    button_style="info",
)

deploy_button = widgets.Button(
    description="Deploy and Reload",
    button_style="warning",
    disabled=True,
)


In [ ]:
def active_zone():
    return zone_selector.value


def active_protocol():
    zone = active_zone()
    protocol = protocol_selector.value
    if protocol not in ZONE_CONFIG[zone]:
        raise ValueError(
            f"Zone {zone} does not define protocol {protocol!r}."
        )
    return protocol


def active_group_index():
    index = group_selector.value
    groups = zone_protocol_config(
        active_zone(), active_protocol()
    )["groups"]
    if index is None or index >= len(groups):
        raise ValueError("Select a configured rule group.")
    return index


def active_rules():
    return rules_by_zone_protocol[active_zone()][active_protocol()][
        active_group_index()
    ]


def set_status(message, error=False):
    with status_output:
        clear_output(wait=True)
        color = "#b91c1c" if error else "#166534"
        display(HTML(f"<p style='color:{color}'>{message}</p>"))


In [ ]:
def refresh_editor(*_):
    zone = active_zone()
    protocol = active_protocol()
    protocol_config = zone_protocol_config(zone, protocol)
    group_index = active_group_index()
    group = protocol_config["groups"][group_index]
    fields = group_fields(group)
    rules = active_rules()

    field_inputs.clear()
    field_inputs.update({
        field: widgets.Text(
            description=f"{field.replace('_', ' ').title()}:",
            layout=widgets.Layout(width="220px"),
        )
        for field in fields
    })
    field_editor.children = tuple(field_inputs.values()) + (verdict_input,)

    rule_selector.options = [
        (
            f"Rule {i}: "
            + ", ".join(
                str(rule[field]) for field in fields[:2]
            ),
            i - 1,
        )
        for i, rule in enumerate(rules, start=1)
    ]

    with table_output:
        clear_output(wait=True)
        rows = [
            {"#": number, **rule}
            for number, rule in enumerate(rules, start=1)
        ]
        columns = ["#"] + list(fields) + ["verdict"]
        display(Markdown(
            f"### Zone {zone} — {protocol_config['label']} — "
            f"{group['label']}"
        ))
        display(pd.DataFrame(rows, columns=columns) if rows else "No rules defined.")

    load_selected_rule()


In [ ]:
def load_selected_rule(*_):
    index = rule_selector.value
    rules = active_rules()

    if index is None or index >= len(rules):
        for widget in field_inputs.values():
            widget.value = ""
        verdict_input.value = "accept"
        return

    rule = rules[index]
    for field, widget in field_inputs.items():
        widget.value = rule[field]
    verdict_input.value = rule["verdict"]


In [ ]:
def form_rule():
    rule = {
        field: widget.value.strip()
        for field, widget in field_inputs.items()
    }
    rule["verdict"] = verdict_input.value

    for field, value in rule.items():
        if field == "verdict":
            continue
        if not value:
            raise ValueError(f"{field.replace('_', ' ')} cannot be empty.")
        if any(c in value for c in ("\n", "\r", ",", ";", "{", "}")):
            raise ValueError(
                f"Invalid character in {field.replace('_', ' ')}."
            )
        if ":" in value:
            raise ValueError(
                f"Unexpected colon in {field.replace('_', ' ')}."
            )
    return rule


In [ ]:
def add_rule(_):
    try:
        rule = form_rule()
        if rule in active_rules():
            raise ValueError("Duplicate rule.")
        active_rules().append(rule)
        refresh_editor()
        deploy_button.disabled = True
        set_status(
            f"Zone {active_zone()} {active_protocol()} rule added in memory."
        )
    except Exception as exc:
        set_status(str(exc), error=True)


def update_rule(_):
    try:
        index = rule_selector.value
        if index is None:
            raise ValueError("Select a rule.")
        rule = form_rule()
        rules = active_rules()
        if any(i != index and item == rule for i, item in enumerate(rules)):
            raise ValueError("Duplicate rule.")
        rules[index] = rule
        refresh_editor()
        deploy_button.disabled = True
        set_status(
            f"Zone {active_zone()} {active_protocol()} rule updated in memory."
        )
    except Exception as exc:
        set_status(str(exc), error=True)


def delete_rule(_):
    index = rule_selector.value
    if index is None:
        set_status("Select a rule.", error=True)
        return
    active_rules().pop(index)
    refresh_editor()
    deploy_button.disabled = True
    set_status(
        f"Zone {active_zone()} {active_protocol()} rule deleted."
    )


In [ ]:
def update_protocol_options(*_):
    zone = active_zone()
    choices = ZONE_CONFIG[zone]
    protocol_selector.options = [
        (config["label"], key) for key, config in choices.items()
    ]
    available = tuple(choices)
    if protocol_selector.value not in available:
        protocol_selector.value = available[0]
    update_group_options()


def update_group_options(*_):
    protocol = active_protocol()
    groups = zone_protocol_config(active_zone(), protocol)["groups"]
    group_selector.options = [
        (group["label"], index)
        for index, group in enumerate(groups)
    ]
    if group_selector.value not in range(len(groups)):
        group_selector.value = 0
    refresh_editor()


zone_selector.observe(update_protocol_options, names="value")
protocol_selector.observe(update_group_options, names="value")
group_selector.observe(refresh_editor, names="value")
rule_selector.observe(load_selected_rule, names="value")
add_button.on_click(add_rule)
update_button.on_click(update_rule)
delete_button.on_click(delete_rule)

editor_header = widgets.HTML("<h2 style='margin: 0 0 8px'>Zone Editor</h2>")
editor_controls = widgets.HBox([
    zone_selector, protocol_selector, group_selector, rule_selector
])
editor_buttons = widgets.HBox([add_button, update_button, delete_button])
workflow_controls = widgets.HBox([stage_button, deploy_button])

display(widgets.VBox([
    editor_header,
    editor_controls,
    field_editor,
    editor_buttons,
    table_output,
    status_output,
    widgets.HTML("<hr><h3>Validate and deploy</h3>"),
    workflow_controls,
    workflow_output,
    diff_output,
]))

update_protocol_options()


In [ ]:
# ============================================================
# Stage all configured zones and protocols
# ============================================================

def build_staged_source(protocol):
    config = PROTOCOLS[protocol]
    if read_source(protocol) != original_sources[protocol]:
        raise RuntimeError(
            f"{protocol} source changed externally. "
            "Reload the notebook before proceeding."
        )

    staged = original_sources[protocol]
    replacements = {}
    for zone, zone_config in ZONE_CONFIG.items():
        if protocol not in zone_config:
            continue
        groups = zone_config[protocol]["groups"]
        for group, rules in zip(
            groups, rules_by_zone_protocol[zone][protocol]
        ):
            for item in group["definitions"]:
                if item["name"] in replacements:
                    raise ValueError(
                        f"Definition {item['name']} appears more than once "
                        "in the zone manifest."
                    )
                replacements[item["name"]] = render_definition(
                    item, rules, zone
                )

    for name, replacement in replacements.items():
        pattern = (
            r"(?m)^[ \t]*define[ \t]+"
            + re.escape(name)
            + r"[ \t]*=[ \t]*\{[^}]*\}"
        )
        staged, count = re.subn(
            pattern,
            lambda match, replacement=replacement: replacement,
            staged,
            count=1,
            flags=re.DOTALL,
        )
        if count != 1:
            raise RuntimeError(
                f"Expected one definition for {name}; found {count}."
            )
    return staged


def stage_all_protocols():
    global staged_sources
    errors = []
    for zone, zone_config in ZONE_CONFIG.items():
        for protocol in zone_config:
            errors.extend(validate_protocol(zone, protocol))
    if errors:
        raise ValueError("\n".join(errors))

    staged_sources = {}
    for protocol, config in PROTOCOLS.items():
        staged = build_staged_source(protocol)
        config["staging"].write_text(staged)
        staged_sources[protocol] = staged

    TEST_ENTRYPOINT.write_text("\n".join([
        "# Generated candidate ruleset for syntax checking.",
        "flush ruleset",
        f'include "{VARIABLES_INCLUDE}"',
        f'include "{CONSTANTS_INCLUDE}"',
        f'include "{PROTOCOLS["ICMP"]["staging"]}"',
        f'include "{PROTOCOLS["UDP"]["staging"]}"',
        f'include "{PROTOCOLS["TCP"]["staging"]}"',
        f'include "{ZONES_INCLUDE}"',
        "",
    ]))


In [ ]:
# ============================================================
# Candidate syntax check
# ============================================================

def check_candidate():
    command = [
        "ip", "netns", "exec", NAMESPACE,
        "nft", "-c", "-f", str(TEST_ENTRYPOINT),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )

    output = "\n".join(
        value.strip()
        for value in (result.stdout, result.stderr)
        if value.strip()
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Candidate syntax check failed "
            f"(exit {result.returncode}).\n{output}"
        )

    return output or "Candidate passed nft -c."

In [ ]:
# ============================================================
# Display proposed changes
# ============================================================

def show_diffs():
    with diff_output:
        clear_output(wait=True)

        for protocol, config in PROTOCOLS.items():
            display(Markdown(f"### Proposed {protocol} changes"))

            diff = difflib.unified_diff(
                original_sources[protocol].splitlines(),
                staged_sources[protocol].splitlines(),
                fromfile=str(config["source"]),
                tofile=str(config["staging"]),
                lineterm="",
            )

            print("\n".join(diff) or "No changes.")

In [ ]:
# ============================================================
# Check button
# ============================================================

def run_check(_):
    deploy_button.disabled = True

    with workflow_output:
        clear_output(wait=True)

        try:
            stage_all_protocols()
            output = check_candidate()
            show_diffs()

            print(f"Zones {', '.join(ZONE_IDS)} were staged.")
            print("All configured protocol candidates passed syntax check.")
            print(f"Test entry point: {TEST_ENTRYPOINT}")
            print(output)

            deploy_button.disabled = False
            set_status("Candidate passed syntax check.")

        except Exception as exc:
            set_status(str(exc), error=True)
            print(f"ERROR: {exc}")


stage_button.on_click(run_check)

In [ ]:
# ============================================================
# Deployment and production reload
# ============================================================

def deploy_all(_):
    deploy_button.disabled = True

    with workflow_output:
        clear_output(wait=True)

        try:
            stage_all_protocols()
            output = check_candidate()
            print(output)

            if not PRODUCTION_ENTRYPOINT.is_file():
                raise FileNotFoundError(
                    f"Missing entry point: {PRODUCTION_ENTRYPOINT}"
                )

            for protocol in PROTOCOLS:
                if read_source(protocol) != original_sources[protocol]:
                    raise RuntimeError(
                        f"{protocol} source changed externally. "
                        "Deployment cancelled."
                    )

            for protocol, config in PROTOCOLS.items():
                shutil.copyfile(
                    config["staging"],
                    config["source"],
                )
                print(f"Deployed {protocol}: {config['source']}")

            command = [
                "ip", "netns", "exec", NAMESPACE,
                "nft", "-f", str(PRODUCTION_ENTRYPOINT),
            ]

            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                timeout=30,
                check=False,
            )

            reload_output = "\n".join(
                value.strip()
                for value in (result.stdout, result.stderr)
                if value.strip()
            )

            if result.returncode != 0:
                raise RuntimeError(
                    "Production reload failed after file replacement. "
                    "The files remain changed.\n" + reload_output
                )

            for protocol in PROTOCOLS:
                original_sources[protocol] = read_source(protocol)

            staged_sources.clear()

            print(
                f"Reloaded {PRODUCTION_ENTRYPOINT} "
                f"in namespace {NAMESPACE}."
            )
            print(reload_output or "Reload succeeded.")

            set_status(
                f"Zones {', '.join(ZONE_IDS)} deployed; ruleset reloaded."
            )

        except Exception as exc:
            print(f"ERROR: {exc}")
            set_status(str(exc), error=True)


deploy_button.on_click(deploy_all)